# Fase 2 — Representación vectorial y unión de características

Construye las 3 representaciones que exige el enunciado — **(1) BoW solo, (2) características léxicas solas, (3) unión** — usando las 24 características validadas en la Fase 0 más las 4 nuevas de la Fase 1. El escalado usa `RobustScaler` (justificado con evidencia más abajo, en vez de `StandardScaler`), y todo transformador se ajusta (`fit`) solo con el conjunto de entrenamiento.


In [1]:
import sys, re, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from nltk import TweetTokenizer

sys.path.insert(0, str(Path.cwd().parent))
from logic.text_processing import TextProcessing
from logic.feature_extraction import FeatureExtraction

SEED = 42
np.random.seed(SEED)


## 1. Carga de datos (TASS 2018 real)

In [2]:
DIR_DATA = Path.cwd().parent / 'data' / 'tass'
TRAIN_FILE, TEST_FILE = "tass2018_es_train.csv", "tass2018_es_test.csv"
COL_TEXT, COL_LABEL = "content", "sentiment/polarity/value"

data_train = pd.read_csv(DIR_DATA / TRAIN_FILE)
data_test = pd.read_csv(DIR_DATA / TEST_FILE)
print(f"Train: {len(data_train)} filas | Test: {len(data_test)} filas")


Train: 1008 filas | Test: 506 filas


## 2. Columnas de texto necesarias

- **`clean`**: salida de `TextProcessing.transformer()` (la misma limpieza que usa `get_features_lexical`).
- **`raw`**: texto original, sin tocar — lo necesitan `elongated_words_count` y `punct_repetition`, porque `remove_patterns()` elimina la puntuación y no distingue elongaciones antes de que estas características puedan verlas.


In [3]:
def limpiar(texto):
    return TextProcessing.transformer(str(texto)) or ""

def construir_columnas(df):
    raw = df[COL_TEXT].astype(str)
    return pd.DataFrame({"raw": raw.values, "clean": raw.map(limpiar).values})

X_train_txt = construir_columnas(data_train)
X_test_txt = construir_columnas(data_test)
y_train = data_train[COL_LABEL].values
y_test = data_test[COL_LABEL].values
X_train_txt.head(3)


,raw,clean
0,-Me caes muy bien \n-Tienes que jugar más part...,me caes muy bien tienes que jugar mas partidas...
1,@myendlesshazza a. que puto mal escribo\n\nb. ...,mention a. que puto mal escribo b. me sigo sur...
2,@estherct209 jajajaja la tuya y la d mucha gen...,mention jajajaja la tuya y la d mucha gente se...


## 3. Adaptador léxico compatible con scikit-learn

Recibe 2 columnas (`raw`, `clean`) y devuelve 28 columnas numéricas: las 24 características de `get_features_lexical()` que sobrevivieron la auditoría de la Fase 0, más las 4 nuevas de la Fase 1 (`lexical_diversity_mattr`, `elongated_words_count`, `punct_repetition`, `lex_pol_neg`).


In [4]:
FEATURE_NAMES_ORIGINALES = [
    "weighted_position", "weighted_normalized", "label_mention", "label_url", "label_hashtag",
    "label_emoji", "label_retweets", "lexical_diversity", "label_word",
    "first_person_singular", "second_person_singular", "third_person_singular",
    "first_person_plurar", "second_person_plurar", "third_person_plurar",
    "avg_word", "kur_word", "skew_word",
    "adverb_neg", "adverb_time", "adverb_place", "adverb_mode", "adverb_cant", "adverb_all",
    "adjetives_neg", "adjetives_pos", "who_general", "who_male", "who_female",
]

EXCLUIDAS_FASE0 = {"weighted_position", "weighted_normalized", "label_word", "lexical_diversity", "adverb_all"}
FEATURE_NAMES_VALIDADAS = [f for f in FEATURE_NAMES_ORIGINALES if f not in EXCLUIDAS_FASE0]
FEATURE_NAMES_NUEVAS = ["mattr", "elongated_words", "punct_repetition", "lex_pol_neg"]
FEATURE_NAMES_FINAL = FEATURE_NAMES_VALIDADAS + FEATURE_NAMES_NUEVAS

print(f"Features validadas de Fase 0: {len(FEATURE_NAMES_VALIDADAS)}")
print(f"Features nuevas de Fase 1:    {len(FEATURE_NAMES_NUEVAS)}")
print(f"Total en el vector lexico:    {len(FEATURE_NAMES_FINAL)}")


class LexicalVectorizer(BaseEstimator, TransformerMixin):
    """Adaptador fit/transform de scikit-learn. Espera 2 columnas: [raw, clean]."""

    def __init__(self, lang="es"):
        self.lang = lang

    def fit(self, X, y=None):
        self.fe_ = FeatureExtraction(self.lang)
        self.tt_ = TweetTokenizer()
        self._idx_validas = [FEATURE_NAMES_ORIGINALES.index(f) for f in FEATURE_NAMES_VALIDADAS]
        return self

    def transform(self, X):
        if not hasattr(self, "fe_"):
            self.fit(X)
        arr = np.asarray(X, dtype=object)
        filas = []
        for raw_txt, clean_txt in zip(arr[:, 0], arr[:, 1]):
            raw_txt = raw_txt if isinstance(raw_txt, str) else ""
            clean_txt = clean_txt if isinstance(clean_txt, str) else ""
            try:
                v = self.fe_.get_features_lexical(clean_txt)
                v = np.asarray(v, dtype=np.float64) if v is not None else None
            except Exception:
                v = None
            if v is None or v.shape[0] != 29 or not np.isfinite(v).all():
                v = np.zeros(29)
            validas = v[self._idx_validas]

            tokens = self.tt_.tokenize(clean_txt) if clean_txt else []
            mattr = FeatureExtraction.lexical_diversity_mattr(tokens, window=10)
            elong = FeatureExtraction.elongated_words_count(raw_txt)
            punct = FeatureExtraction.punct_repetition(raw_txt)
            pol = FeatureExtraction.lex_pol_neg(tokens)

            filas.append(np.concatenate([validas, [mattr, elong, punct, pol]]))
        return np.vstack(filas)


Features validadas de Fase 0: 24
Features nuevas de Fase 1:    4
Total en el vector lexico:    28


## 4. Justificación del escalado: `RobustScaler`, no `StandardScaler`

Varias características nuevas son conteos raros con muchos ceros (`punct_repetition`: 95% ceros; `elongated_words_count`: 97% ceros), con sesgo (*skewness*) superior a 6 en ambos casos. `StandardScaler` centra con la media, lo que convierte el valor 0 (el caso normal, mayoritario) en un número negativo distinto de cero — borrando la señal de "esto no ocurrió". `RobustScaler` usa mediana e IQR, insensibles a esos valores raros y extremos, y preserva 0→0 en el caso típico.


In [5]:
from scipy import stats

lex_probe = LexicalVectorizer().fit(X_train_txt[["raw", "clean"]].values)
lex_vals_train = lex_probe.transform(X_train_txt[["raw", "clean"]].values)

idx_punct = FEATURE_NAMES_FINAL.index("punct_repetition")
idx_elong = FEATURE_NAMES_FINAL.index("elongated_words")
for nombre, idx in [("punct_repetition", idx_punct), ("elongated_words", idx_elong)]:
    col = lex_vals_train[:, idx]
    print(f"{nombre}: skew={stats.skew(col):.2f}  %ceros={100*np.mean(col==0):.1f}%")


punct_repetition: skew=6.82  %ceros=95.1%
elongated_words: skew=6.42  %ceros=97.0%


### Validación empírica: ¿el escalador elegido también ayuda al modelo, no solo en teoría?

La justificación de arriba es estadística (distribución de los datos). Para confirmar que no es solo teórico, se entrena un modelo simple (regresión logística) sobre **E1 — solo léxico**, comparando `RobustScaler` contra `StandardScaler`, con validación cruzada estratificada de 5 folds sobre el mismo split de train (nunca se toca test aquí). Esto NO es la selección final de clasificador — esa es tarea de la Fase 3 — es solo la confirmación de que la elección del escalador hecha en esta fase también se sostiene con un modelo real.


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

def make_lex_con_escalador(scaler):
    return ColumnTransformer([
        ("lex", Pipeline([("vec", LexicalVectorizer()), ("scale", scaler)]), ["raw", "clean"])
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
resultados_escalador = {}

for nombre_scaler, scaler in [("RobustScaler", RobustScaler()), ("StandardScaler", StandardScaler())]:
    pipe = Pipeline([
        ("rep", make_lex_con_escalador(scaler)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
    ])
    scores = cross_val_score(pipe, X_train_txt[["raw", "clean"]], y_train, cv=cv, scoring="f1_macro")
    resultados_escalador[nombre_scaler] = scores
    print(f"{nombre_scaler:16s} -> macro-F1 CV: {scores.mean():.4f} (+/- {scores.std():.4f})  folds={np.round(scores,4)}")

diferencia = resultados_escalador["RobustScaler"].mean() - resultados_escalador["StandardScaler"].mean()
print(f"\nDiferencia (RobustScaler - StandardScaler): {diferencia:+.4f}")


RobustScaler     -> macro-F1 CV: 0.3662 (+/- 0.0144)  folds=[0.3449 0.3582 0.3731 0.3879 0.3671]
StandardScaler   -> macro-F1 CV: 0.3589 (+/- 0.0161)  folds=[0.336  0.3449 0.3736 0.3777 0.3624]

Diferencia (RobustScaler - StandardScaler): +0.0073


## 5. Las 3 representaciones (fit SOLO sobre train)

- **E0 — BoW solo:** `CountVectorizer` sobre `clean`.
- **E1 — Léxico solo:** `LexicalVectorizer` + `RobustScaler`, sin ningún texto crudo/BoW.
- **E2 — Unión:** BoW + léxico, combinados con `ColumnTransformer`.

`ColumnTransformer.fit_transform` se llama únicamente sobre `X_train_txt`; `X_test_txt` solo pasa por `.transform()` — nunca se ajusta ningún transformador con datos de test.


In [7]:
def make_bow():
    return ColumnTransformer([("bow", CountVectorizer(analyzer="word", ngram_range=(1, 3)), "clean")])

def make_lex():
    return ColumnTransformer([
        ("lex", Pipeline([("vec", LexicalVectorizer()), ("scale", RobustScaler())]), ["raw", "clean"])
    ])

def make_union():
    return ColumnTransformer([
        ("bow", CountVectorizer(analyzer="word", ngram_range=(1, 3)), "clean"),
        ("lex", Pipeline([("vec", LexicalVectorizer()), ("scale", RobustScaler())]), ["raw", "clean"]),
    ], sparse_threshold=0.3)

REPRESENTACIONES = {
    "E0 - BoW solo": make_bow(),
    "E1 - Lexico solo": make_lex(),
    "E2 - Union (BoW + lexico)": make_union(),
}

dimensiones = {}
for nombre, ct in REPRESENTACIONES.items():
    X_tr = ct.fit_transform(X_train_txt[["raw", "clean"]], y_train)   # fit SOLO en train
    X_te = ct.transform(X_test_txt[["raw", "clean"]])                  # test solo transform
    dimensiones[nombre] = X_tr.shape[1]
    print(f"{nombre:30s} -> dimension: {X_tr.shape[1]:6d}  (train: {X_tr.shape[0]} x {X_tr.shape[1]}, test: {X_te.shape[0]} x {X_te.shape[1]})")


E0 - BoW solo                  -> dimension:  26057  (train: 1008 x 26057, test: 506 x 26057)
E1 - Lexico solo               -> dimension:     28  (train: 1008 x 28, test: 506 x 28)
E2 - Union (BoW + lexico)      -> dimension:  26085  (train: 1008 x 26085, test: 506 x 26085)


## 6. ¿Qué aporta y qué pierde cada representación?

| Representación | Dimensión | Qué aporta | Qué pierde |
|---|---|---|---|
| **E0 — BoW solo** | Miles (vocabulario × n-gramas 1-3) | Vocabulario específico: qué palabras exactas aparecen | Intensidad/tono: no distingue "bien" de "BIEN!!!" ni resuelve negación |
| **E1 — Léxico solo** | 28 (fijo, no depende del vocabulario) | Señales de forma e intensidad: negación resuelta, elongación, puntuación repetida, personas gramaticales | Vocabulario: no sabe *qué* palabras específicas se usaron, solo patrones generales |
| **E2 — Unión** | BoW + 28 | Lo mejor de ambas: vocabulario específico + señales de intensidad/negación | Mayor dimensionalidad → más riesgo de sobreajuste con pocos datos; escalas mixtas (disperso de BoW + denso léxico) que exigen manejo cuidadoso (por eso el escalador) |

La diferencia de dimensión entre E0/E1 y E2 depende del tamaño del vocabulario real de TASS (columna `bow` variable, columna `lex` siempre fija en 28) — se calcula explícitamente arriba con datos reales, no un estimado.


## 7. Prueba de que no hay fuga: el escalador NO se ajusta con test

In [8]:
def test_fit_solo_en_train():
    ct_lex = make_lex()
    ct_lex.fit(X_train_txt[["raw", "clean"]], y_train)
    scaler_ajustado = ct_lex.named_transformers_["lex"].named_steps["scale"]
    center_train = scaler_ajustado.center_.copy()

    # Si "por error" se reajustara con test, el centro cambiaria. Verificamos que transform() no lo modifica.
    _ = ct_lex.transform(X_test_txt[["raw", "clean"]])
    assert np.allclose(scaler_ajustado.center_, center_train), "El escalador no deberia cambiar al transformar test"
    print("OK: el RobustScaler conserva los parametros de train al transformar test (sin fuga)")

test_fit_solo_en_train()


OK: el RobustScaler conserva los parametros de train al transformar test (sin fuga)
